# 🫁 Respiratory Sound Classification — Deep Learning Experiments
## Stage 5 & 6 — Model Experiments, Ablation & Evaluation

> **Objective:** Train and evaluate CNN-LSTM architectures for respiratory sound classification on ICBHI 2017 dataset.
> **Classes:** Normal (0), Crackle (1), Wheeze (2), Both (3)
> **Hardware Acceleration:** CUDA / GPU enabled
> **Notice:** Educational & Research purposes only. NOT for clinical diagnosis.

---

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Set seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))
from model import build_model, CNNLSTMClassifier, CLASS_NAMES, NUM_CLASSES
from dataset_loader import get_dataloaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')

### 1. Load DataLoaders (Patient-Level Split, Zero Leakage)

In [ ]:
train_loader, val_loader, test_loader, class_weights = get_dataloaders(
    batch_size=32, feature_key='mel', num_workers=0, seed=SEED
)
print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')
print(f'Class weights: {class_weights}')

### 2. Inspect CNN-LSTM Model Architecture & Parameter Count

In [ ]:
model, _ = build_model(device=str(device))
print(f'Total trainable parameters: {model.count_parameters():,}')

# Sample batch forward pass
sample_x, sample_y = next(iter(train_loader))
sample_x = sample_x.to(device)
with torch.no_grad():
    logits = model(sample_x)
    probs = F.softmax(logits, dim=-1)

print(f'Batch input shape : {sample_x.shape}')
print(f'Batch output shape: {logits.shape}')
print(f'Sample output probabilities:\n{probs[0].cpu().numpy()}')

### 3. Load Trained Model & Evaluation Artifacts

In [ ]:
checkpoint_path = '../models/best_model.pth'
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    print(f'✅ Checkpoint loaded from epoch {ckpt["epoch"]} (val_loss: {ckpt["val_loss"]:.4f})')
else:
    print('⚠️ Checkpoint not found. Train first with python src/train.py')

### 4. Evaluate Test Set & Plot Confusion Matrix

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        p = F.softmax(model(x), dim=-1).cpu().numpy()
        all_probs.append(p)
        all_preds.extend(np.argmax(p, axis=1))
        all_labels.extend(y.numpy())

all_probs = np.vstack(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# Confusion Matrix Plot
cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Test Set Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()

### 5. Multi-Class ROC Curves

In [ ]:
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
fig, ax = plt.subplots(figsize=(8, 6))

for cls_idx, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    y_bin = (all_labels == cls_idx).astype(int)
    if y_bin.sum() > 0:
        fpr, tpr, _ = roc_curve(y_bin, all_probs[:, cls_idx])
        auc = roc_auc_score(y_bin, all_probs[:, cls_idx])
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (One-vs-Rest) — Test Set', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.show()